<a href="https://colab.research.google.com/github/Nabamit/Antigravity_project/blob/main/LangChain_Agent_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install LangChain and the official Google GenAI integration
!pip install -q -U langchain langchain-google-genai langchain-core

In [ ]:
import os
import getpass

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")

### Using Google Gemini as an alternative LLM

Since you have `langchain-google-genai` installed and your `GOOGLE_API_KEY` is already configured, you can switch to using a Gemini model for your agent. We will define the `model` variable using `ChatGoogleGenerativeAI`.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Explicitly pass the active model name
model = ChatGoogleGenerativeAI(
    model="gemma-4-26b-a4b-it",
    temperature=0
)

In [ ]:
import math
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together. Use for addition operations."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together. Use for multiplication operations."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second number. Use for division operations."""
    if b == 0:
        return "Error: Division by zero is undefined."
    return a / b

@tool
def square_root(n: float) -> float:
    """Calculate the square root of a number."""
    return math.sqrt(n)

# Register the 4 tools
tools = [add, multiply, divide, square_root]

In [ ]:
from langchain.agents import create_agent

# Build the ReAct reasoning loop
agent = create_agent(
    model=model,
    tools=tools,
)

In [ ]:
def run_agent(question: str):
    """Run the agent and print the execution trace."""
    print(f"🧑 User: {question}")
    print("-" * 50)

    result = agent.invoke({
        "messages": [("user", question)]
    })

    print("🤖 Agent:", result["messages"][-1].content)
    print("\n" + "=" * 50 + "\n")

In [ ]:
# Test 1: Single tool call (Addition)
run_agent("What is 42 + 58?")

# Test 2: Sequential calls (Multiplication -> Division)
run_agent("What is 15 multiplied by 8, then divided by 3?")

# Test 3: Multi-step reasoning
run_agent(
    "I have a rectangle with width 12 and height 7. "
    "What is its area, and what is the square root of that area?"
)

🧑 User: What is 42 + 58?
--------------------------------------------------
🤖 Agent: [{'type': 'thinking', 'thinking': 'The user asked for 42 + 58, and the tool returned 100. I should now provide the final answer.'}, {'type': 'text', 'text': '42 + 58 = 100'}]


🧑 User: What is 15 multiplied by 8, then divided by 3?
--------------------------------------------------
🤖 Agent: 15 multiplied by 8 is 120, and 120 divided by 3 is 40.


🧑 User: I have a rectangle with width 12 and height 7. What is its area, and what is the square root of that area?
--------------------------------------------------
🤖 Agent: The area of the rectangle is 84, and the square root of the area is approximately 9.17.


